# 00 · Setup and test subgraph

Run this once per machine, after `bash setup/setup_local.sh`. It
1. checks the GPU and DGL (`setup/verify_gpu.py`),
2. extracts the thesis graph from `ddi_graph_and_linegraph.zip` and reads **only the drug graph** (3,342 drugs), so this step needs only a few GB of RAM,
3. builds a random-drug test subgraph for warm start / diagnostics / episodic, and a no-negatives version for cold start,
4. saves both under `WORK_ROOT/test_graphs/`.

The test subgraph is only for checking that the code runs. Its numbers mean nothing.

In [ ]:
DATA_ROOT = "<DATA_ROOT: e.g. /home/you/ddi_data>"   # PLACEHOLDER: thesis inputs (read only)
WORK_ROOT = "<WORK_ROOT: e.g. /home/you/ddi_runs>"   # PLACEHOLDER: new outputs
TEST_MODE = True     # True: small test subgraph from notebook 00, few epochs
                     # False: full thesis graph (needs about 64 GB of RAM)
N_TEST_DRUGS = 800   # must match the test graph built in notebook 00

In [ ]:
import os, sys
# The package is installed in the venv by setup/setup_local.sh (editable install).
# Run this notebook with the "ddi (5090)" kernel.
try:
    import ddi
except ImportError:
    raise SystemExit('ddi is not importable: run "bash setup/setup_local.sh" and select '
                     'the "ddi (5090)" kernel.')
REPO_DIR = os.path.abspath(os.path.join(os.path.dirname(ddi.__file__), "..", ".."))
print("python:", sys.executable)
print("ddi from:", os.path.dirname(ddi.__file__))

In [ ]:
!{sys.executable} {REPO_DIR}/setup/verify_gpu.py

In [ ]:
import json, gc
import numpy as np
import torch
import dgl
import warnings
warnings.filterwarnings("ignore", message="Dataloader CPU affinity opt is not enabled")

from ddi.paths import get_paths
paths = get_paths(DATA_ROOT, WORK_ROOT)
paths.make_output_dirs()
paths.show()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| TEST_MODE:", TEST_MODE)

## Test subgraph
Random drugs → every interaction (and sampled non-interacting pair) among them → directed line graph, same rule as the thesis (`backtracking=False`).

In [ ]:
from ddi import graph as G
from ddi.data import save_graph_pair

G.ensure_graph_bin(paths.graph_zip, paths.graph_bin)
drug_graph = G.load_drug_graph(paths.graph_bin)
print(drug_graph)

SEED = 0
sub, lg = G.make_test_subgraph(drug_graph, n_drugs=N_TEST_DRUGS, seed=SEED)
save_graph_pair(os.path.join(paths.test_graph_dir, f"test_pair_{N_TEST_DRUGS}.bin"), sub, lg)

# cold start uses the graph without the sampled non-interacting pairs
sub_cs, _ = G.make_test_subgraph(drug_graph, n_drugs=N_TEST_DRUGS, seed=SEED, drop_negatives=True)
dgl.save_graphs(os.path.join(paths.test_graph_dir, f"test_drug_graph_noneg_{N_TEST_DRUGS}.bin"), [sub_cs])

In [ ]:
labels = sub.edata["label"]
counts = torch.bincount(labels)
present = (counts > 0).nonzero().flatten()
print(f"classes present: {len(present)} | with >=4 examples: {int((counts >= 4).sum())}")
print(f"label-0 pairs: {int(counts[0])} of {labels.numel()}")
print("META-DDIE groups present  common(1-175): %d  fewer(176-204): %d  rare(205-227): %d" % (
    int(((present >= 1) & (present <= 175)).sum()),
    int(((present >= 176) & (present <= 204)).sum()),
    int(((present >= 205) & (present <= 227)).sum())))
print(f"line graph: {lg.num_nodes():,} nodes, {lg.num_edges():,} edges "
      f"(~{lg.num_edges()*16/1e9:.2f} GB of edge indices)")